In [1]:
# Install necessary libraries
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 16.9 MB/s eta 0:00:00


In [2]:
# Load sensitive information from Colab's user data
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
google_gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

In [3]:
# Import ngrok and os
import os
from pyngrok import ngrok

In [4]:
# Kill any existing ngrok tunnels
ngrok.kill()

In [5]:
# Set up ngrok tunnel and update LINE webhook URL
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://thumping-feline-unkempt.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://thumping-feline-unkempt.ngrok-free.dev


True

In [ ]:
# Flask application for LINE webhook handler
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)

from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    print("EVENT: ", event)
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        line_bot_api.reply_message_with_http_info(
            ReplyMessageRequest(
                reply_token=event.reply_token,
                messages=[TextMessage(text=event.message.text),
                            TextMessage(text=event.message.text)]
            )
        )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 519, in _get_current_object
    raise RuntimeError(unbound_message) from None
RuntimeError: Working outside of request context.

This typically means that you attempted to use functionality that needed
an active HTTP request. Consult the documentation on testing for
information abo

In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "unzqzbjN1ffY",
    "outputId": "aba6c13b-05b7-4d07-f032-4042a5bb51f0"
   },
   "outputs": [],
   "source": [
    "!pip install Flask pyngrok line-bot-sdk requests --quiet\n",
    "!pip install google-genai --quiet"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "xCpJ4xMy1ffa"
   },
   "outputs": [],
   "source": [
    "from google.colab import userdata\n",
    "\n",
    "ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')\n",
    "line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')\n",
    "line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')\n",
    "google_gemini_api_key = userdata.get('GEMINI_API_KEY')\n",
    "port = 5051"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "eTvmRLbj1ffb"
   },
   "outputs": [],
   "source": [
    "import os\n",
    "from pyngrok import ngrok"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ngrok.kill()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "xstUJCaQ1ffb",
    "outputId": "7e664945-59e6-4423-f69e-e856e3693853"
   },
   "outputs": [],
   "source": [
    "import requests\n",
    "\n",
    "ngrok.set_auth_token(ngrok_authtoken)\n",
    "tunnel = ngrok.connect(5051, name=\"linebot_tunnel\")\n",
    "webhook_url = tunnel.public_url\n",
    "\n",
    "print(f\"Ngrok URL: {webhook_url}\")\n",
    "\n",
    "# 自動更新 LINE Webhook URL\n",
    "def update_line_webhook(webhook_url):\n",
    "    \"\"\"使用 LINE Messaging API 更新 Webhook URL\"\"\"\n",
    "    url = \"https://api.line.me/v2/bot/channel/webhook/endpoint\"\n",
    "    headers = {\n",
    "        \"Authorization\": f\"Bearer {line_channel_access_token}\",\n",
    "        \"Content-Type\": \"application/json\"\n",
    "    }\n",
    "    data = {\n",
    "        \"endpoint\": webhook_url\n",
    "    }\n",
    "    \n",
    "    response = requests.put(url, headers=headers, json=data)\n",
    "    \n",
    "    if response.status_code == 200:\n",
    "        print(f\"✅ LINE Webhook URL 已自動更新為：{webhook_url}\")\n",
    "        return True\n",
    "    else:\n",
    "        print(f\"❌ 更新失敗：{response.status_code} - {response.text}\")\n",
    "        return False\n",
    "\n",
    "# 執行更新\n",
    "update_line_webhook(webhook_url)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "background_save": true,
     "base_uri": "https://localhost:8080/"
    },
    "id": "Phvh3eaP1ffb",
    "outputId": "a8a89f54-c163-4bff-ed64-8a3b8b31f705"
   },
   "outputs": [],
   "source": [
    "from flask import Flask, request, abort\n",
    "\n",
    "from linebot.v3 import (\n",
    "    WebhookHandler\n",
    ")\n",
    "from linebot.v3.exceptions import (\n",
    "    InvalidSignatureError\n",
    ")\n",
    "from linebot.v3.messaging import (\n",
    "    Configuration,\n",
    "    ApiClient,\n",
    "    MessagingApi,\n",
    "    ReplyMessageRequest,\n",
    "    TextMessage,\n",
    ")\n",
    "\n",
    "from linebot.v3.webhooks import (\n",
    "    MessageEvent,\n",
    "    TextMessageContent,\n",
    ")\n",
    "\n",
    "app = Flask(__name__)\n",
    "\n",
    "configuration = Configuration(access_token=line_channel_access_token)\n",
    "handler = WebhookHandler(line_channel_secret)\n",
    "\n",
    "\n",
    "@app.route(\"/\", methods=['POST'])\n",
    "def callback():\n",
    "    # get X-Line-Signature header value\n",
    "    signature = request.headers['X-Line-Signature']\n",
    "\n",
    "    # get request body as text\n",
    "    body = request.get_data(as_text=True)\n",
    "    print(\"BODY: \", body)\n",
    "    app.logger.info(\"Request body: \" + body)\n",
    "\n",
    "    # handle webhook body\n",
    "    try:\n",
    "        handler.handle(body, signature)\n",
    "    except InvalidSignatureError:\n",
    "        app.logger.info(\"Invalid signature. Please check your channel access token/channel secret.\")\n",
    "        abort(400)\n",
    "\n",
    "    return 'OK'\n",
    "\n",
    "\n",
    "@handler.add(MessageEvent, message=TextMessageContent)\n",
    "def handle_message(event):\n",
    "    print(\"EVENT: \", event)\n",
    "    with ApiClient(configuration) as api_client:\n",
    "        line_bot_api = MessagingApi(api_client)\n",
    "        line_bot_api.reply_message_with_http_info(\n",
    "            ReplyMessageRequest(\n",
    "                reply_token=event.reply_token,\n",
    "                messages=[TextMessage(text=event.message.text),\n",
    "                            TextMessage(text=event.message.text)]\n",
    "            )\n",
    "        )\n",
    "\n",
    "if __name__ == \"__main__\":\n",
    "    app.run(port=port)"
   ]
  }
 ],
 "metadata": {
  "colab": {
   "provenance": []
  },
  "kernelspec": {
   "display_name": "linebot",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.12.3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 0
}